In [1]:
!pip uninstall -y langchain -q
!pip install -q \
langchain==0.2.14 \
langchain-community==0.2.12 \
langchain-text-splitters==0.2.2 \
langchain-groq \
faiss-cpu \
sentence-transformers \
pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.8/997.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 require

In [2]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [3]:
from google.colab import files
uploaded = files.upload()

Saving SCIT Handbook MBA (ITBM) 2025.csv to SCIT Handbook MBA (ITBM) 2025.csv


In [4]:
from google.colab import files
uploaded = files.upload()

Saving SCIT Handbook MBA-DS&DA 2025.csv to SCIT Handbook MBA-DS&DA 2025.csv


In [8]:
# Clean reinstall core scientific stack
!pip uninstall -y numpy pandas faiss-cpu -q

!pip install numpy==1.26.4 pandas==2.2.2 faiss-cpu -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you ha

In [5]:
import pandas as pd

df1 = pd.read_csv("SCIT Handbook MBA (ITBM) 2025.csv")
df2 = pd.read_csv("SCIT Handbook MBA-DS&DA 2025.csv")

print("ITBM columns:", df1.columns)
print("DSDA columns:", df2.columns)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 263, saw 2


In [32]:
import pandas as pd
from langchain_core.documents import Document

df1 = pd.read_csv("SCIT Handbook MBA (ITBM) 2025.csv", sep=',', engine='python', on_bad_lines='skip')
df2 = pd.read_csv("SCIT Handbook MBA-DS&DA 2025.csv", sep=',', engine='python', on_bad_lines='skip')

# Strip whitespace from column names
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

# Extract the relevant content column from each dataframe and rename to a common 'content' column
df1_content = df1['Student Handbook 2025'].to_frame(name='content')
df2_content = df2['Student Handbook 2025 1'].to_frame(name='content') # Assuming 'Student Handbook 2025 1' is the correct column in df2

# Concatenate the content columns
df_combined_content = pd.concat([df1_content, df2_content], ignore_index=True)

documents = [
    Document(page_content=row["content"])
    for _, row in df_combined_content.iterrows()
    if pd.notna(row["content"])
]

print("Total documents:", len(documents))

Total documents: 2944


In [9]:
documents = [
    Document(page_content=" ".join(map(str, row)))
    for row in df.values
]

In [10]:
print(df.columns)

Index(['   Student Handbook 2025 ', 'Student Handbook 2025 1 '], dtype='object')


In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 2946


In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipython-input-3419/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [45]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever()

In [16]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [17]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama3-8b-8192",
    temperature=0
)

In [19]:
import os
print(os.environ.get("GROQ_API_KEY"))

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


In [21]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [22]:
print(llm.invoke("Hello").content)

Hello. How can I assist you today?


In [24]:
def simple_rag(query):

    # Retrieve relevant chunks
    docs = retriever.get_relevant_documents(query)

    # Combine retrieved text
    context = "\n\n".join([doc.page_content for doc in docs])

    # Create prompt
    prompt = f"""
    You are an AI assistant for SCIT students.
    Answer only from the provided context.
    If not found, say 'Not available in handbook.'

    Context:
    {context}

    Question: {query}
    """

    # Generate answer
    response = llm.invoke(prompt)

    return response.content

In [25]:
print(simple_rag("What is the attendance requirement at SCIT?"))

/tmp/ipython-input-3419/1898456499.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents(query)


Minimum 75% attendance is mandatory before taking admission into SCIT.


In [47]:
print(simple_rag("What is the minimum attendance requirement at SCIT?"))
print(simple_rag("What happens if attendance is below required percentage?"))
print(simple_rag("Is 90 percent attendance required for internship eligibility?"))
print(simple_rag("Explain the examination evaluation pattern at SCIT."))

Minimum 75% attendance is mandatory at SCIT.
Course Not Granted (CNG): Minimum 75% attendance is mandatory.
Not available in handbook.
According to the provided context, the examination evaluation pattern at SCIT involves a continuous evaluation process that demands an Internal Assessment Test (CIAT).


In [27]:
print(simple_rag("What is the minimum attendance percentage required to appear for semester examinations at SCIT?"))

print(simple_rag("Is 90 percent attendance required for internship eligibility?"))

print(simple_rag("What happens if a student's attendance falls below the required limit?"))

print(simple_rag("How are medical leaves treated in attendance calculation?"))

Minimum 75% attendance is mandatory.
Not available in handbook.
If a student's attendance falls below the required limit, they will not be allowed to attend the classes/lab for the concerned course.
Not available in handbook.


In [28]:
print(simple_rag("What is the minimum attendance percentage required at SCIT?"))

print(simple_rag("Is attendance mandatory for all courses?"))

print(simple_rag("What is the consequence of attendance shortage?"))

Minimum 75% attendance is mandatory for a particular course.
Not available in handbook.
Not available in handbook.


In [29]:
print(simple_rag("How many semesters are there in the MBA program at SCIT?"))

print(simple_rag("How many credits are required to complete the MBA program?"))

print(simple_rag("What is the duration of the MBA program?"))

Not available in handbook.
Not available in handbook.
Not available in handbook.


In [30]:
print(df.head())
print(df.columns)
print(len(df))
print(df.iloc[0])

                              Student Handbook 2025  Student Handbook 2025 1 
0                            Student Hand Book 2025                       NaN
1                                          MBA-ITBM                       NaN
2  Plot No. P-15, M.I.D.C, Symbiosis Infotech Cam...                      NaN
3  Hinjawadi, Pune -411057.Tel No.: 020-22934308/...                      NaN
4                              Website: www.scit.edu                      NaN
Index(['   Student Handbook 2025 ', 'Student Handbook 2025 1 '], dtype='object')
2944
   Student Handbook 2025     Student Hand Book 2025 
Student Handbook 2025 1                          NaN
Name: 0, dtype: object


In [41]:
import pandas as pd
from langchain_core.documents import Document

df1 = pd.read_csv("SCIT Handbook MBA (ITBM) 2025.csv", header=None, sep=',', engine='python', on_bad_lines='skip')
df2 = pd.read_csv("SCIT Handbook MBA-DS&DA 2025.csv", header=None, sep=',', engine='python', on_bad_lines='skip')

df = pd.concat([df1, df2], ignore_index=True)

documents = []

for _, row in df.iterrows():
    # Combine all columns into one text
    full_text = " ".join([str(value) for value in row if pd.notna(value)])

    # Only add if there is actual content, without an arbitrary length filter
    if full_text.strip(): # Check if it's not just whitespace
        documents.append(Document(page_content=full_text))

print("Total cleaned documents:", len(documents))
print("Sample document:\n")
# Print a sample if documents list is not empty
if documents:
    print(documents[0].page_content)
else:
    print("No documents found after cleaning.")

Total cleaned documents: 2946
Sample document:

   Student Handbook 2025 


In [46]:
vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [37]:
print(simple_rag("What is the minimum attendance requirement at SCIT?"))

Not available in handbook.


In [42]:
import pandas as pd
from langchain_core.documents import Document

df1 = pd.read_csv("SCIT Handbook MBA (ITBM) 2025.csv", header=None, sep=',', engine='python', on_bad_lines='skip')
df2 = pd.read_csv("SCIT Handbook MBA-DS&DA 2025.csv", header=None, sep=',', engine='python', on_bad_lines='skip')

df = pd.concat([df1, df2], ignore_index=True)

# Merge ALL content into one large string
all_text = ""

for _, row in df.iterrows():
    full_text = " ".join([str(value) for value in row if pd.notna(value)])
    all_text += full_text + "\n"

print("Total text length:", len(all_text))

Total text length: 124622


In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

docs = text_splitter.create_documents([all_text])

print("Total chunks after proper splitting:", len(docs))

Total chunks after proper splitting: 192


In [48]:
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [49]:
def simple_rag(query):
    retrieved_docs = retriever.get_relevant_documents(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
    Answer the question ONLY using the context below.
    If answer is not present, say "Not available in handbook".

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)
    return response.content

In [50]:
print(simple_rag("What is the minimum attendance requirement at SCIT?"))

print(simple_rag("How many semesters are there in the MBA program?"))

print(simple_rag("What grading system is followed at SCIT?"))

print(simple_rag("Is summer internship mandatory?"))

75% or above.
Not available in handbook.
The system of 9 grades will be applicable for the students admitted during academic year 2016. The grade points corresponding to 9 grades will be as follows:

Letter Grade Proportion Grade Point 
O (Outstanding) Top 3% 10 
A+ (Excellent) 12% 9 
A (Very Good) 21% 8 
B+ (Good) 28% 7 
B (Above Average) 21% 6 
C (Average) 12% 5 
P (Pass) Bottom 3% 4 
F (Fail) 0 
AB (Absent) 0
Not available in handbook.


In [51]:
print(simple_rag("What is the duration of the MBA program at SCIT?"))

Not available in handbook.


In [52]:
print(simple_rag("The MBA program is divided into how many semesters?"))

Not available in handbook.


In [53]:
print(simple_rag("How many credits are required to complete the MBA program?"))

Not available in handbook.


In [54]:
print(simple_rag("Explain the grading system followed at SCIT."))

The grading system followed at SCIT is based on a 10-point grade scale with grades denoted by letters O, A+, A, B+, B, C, P, F, and AB. The grade points corresponding to these grades are as follows:

- O (Outstanding): Top 3% - 10 grade points
- A+ (Excellent): 12% - 9 grade points
- A (Very Good): 21% - 8 grade points
- B+ (Good): 28% - 7 grade points
- B (Above Average): 21% - 6 grade points
- C (Average): 12% - 5 grade points
- P (Pass): Bottom 3% - 4 grade points
- F (Fail): 0 grade points
- AB (Absent): 0 grade points


In [55]:
print(simple_rag("Explain the summer internship guidelines."))

As a part of the curriculum, all students need to work in the industry or work on an industry project, tentatively from April 1, 2025 to June 30, 2025. The Placement Committee will make all possible efforts to acquire internship opportunities in the industry. Students may also try to find internships on their own wherever possible. It should be noted that the responsibility of seeking opportunities lies as much with the individual students as the placement team. The internship policy will be circulated at the appropriate time.


In [56]:
print(simple_rag("What is the minimum grade required to pass a course?"))

Not available in handbook.


In [57]:
print(simple_rag("What is the duration of the MBA program at SCIT?"))


Not available in handbook.


In [58]:
print(simple_rag("What is the vision of SCIT?"))

Not available in handbook.


In [59]:
print(simple_rag("What is the mission of SCIT?"))

Not available in handbook.


In [60]:
print(simple_rag("Explain the grading system in detail."))

The grading system in the given context is as follows:

1. **Grading Chart**: The grading chart is based on a 10-point grade scale with grades denoted by letters O, A+, A, B+, B, C, P, F, and AB.

2. **Letter Grade Proportion Grade Point**:
   - O (Outstanding): Top 3% - 10 grade points
   - A+ (Excellent): 12% - 9 grade points
   - A (Very Good): 21% - 8 grade points
   - B+ (Good): 28% - 7 grade points
   - B (Above Average): 21% - 6 grade points
   - C (Average): 12% - 5 grade points
   - P (Pass): Bottom 3% - 4 grade points
   - F (Fail): 0 grade points
   - AB (Absent): 0 grade points

3. **Weighted Total of Grade Points**: The weighted total of grade points for a course is worked out by considering the internal (60%) and external (40%) grades. The final grade is given based on this weighted total.

4. **Standard of Passing**: A student has to pass both internal and external exams separately. The grade 'F' of individual head (internal and external) will be considered as fail.

5. 

In [61]:
print(simple_rag("What are the general rules regarding attendance?"))

According to the context, the general rules regarding attendance are:

1. Students should attend all sessions in their own interest. No in-class evaluation will be repeated.
2. Students should ensure that their attendance in all courses is 75% or above. Defaulters will not be allowed to appear for the exams.
3. Students need to ensure 90% attendance for all the classes to be eligible for the internship & placement processes.
4. All academic absences should be pre-intimated to the respective Program in charge and Academic Coordinator.
5. Attendance will be taken by the warden between 10.00PM to 11.00PM (11.30PM on Sun/Holiday) and students are not allowed to go out of the hostel after marking attendance.


In [62]:
print(simple_rag("What are the academic conduct rules mentioned in the handbook?"))

According to the handbook, the following are the academic conduct rules:

1. 90% attendance in academic sessions.
2. 90% attendance in all guest lectures including those in seminars.
3. No disciplinary violation.
4. Attend all events managed by the Placement Committee such as National Seminar, domain specific Seminar etc.
5. Students have to be courteous to the mess staff.
6. Students are to write complaints / suggestions if any regarding the mess service in the register kept at the counter.
7. No food items, snacks, tea, coffee are to be carried in the academic/ admin block and hostels.
8. General behavior must be polite and courteous to all at all time.
9. Arguments and demands with mess, cafeteria staff, vendors, staff, faculty, in public places or over telephone etc. should be avoided.
10. While in the Academic Block boys should wear formals & the girls should wear a Salwar Kameez or formals (western outfits). No casual wear will be allowed in the Academic Block.


In [63]:
print(simple_rag("How is student performance evaluated in each course?"))

Student performance is evaluated in each course through a combination of internal and external evaluation components. The internal evaluation is carried out by the concerned faculty during a semester and accounts for 60% of the total evaluation, while the external evaluation is conducted by SIU through a semester-end exam and accounts for 40% of the total evaluation.


In [64]:
print(simple_rag("What is the address of SCIT?"))

Not available in handbook.


In [65]:
print(simple_rag("Provide the complete postal address of Symbiosis Centre for Information Technology."))

Not available in handbook.


In [66]:
print(simple_rag("Where is SCIT located?"))

Not available in handbook.


In [67]:
for chunk in docs:
    if "hinjawadi" in chunk.page_content.lower():
        print(chunk.page_content)
        break

Student Handbook 2025 
Student Hand Book 2025 
MBA-ITBM 
Plot No. P-15, M.I.D.C, Symbiosis Infotech Campus, Rajiv Gandhi Infotech Park, 
Hinjawadi, Pune -411057.Tel No.: 020-22934308/09. 
Website: www.scit.edu
 2 
Student Handbook 2025 
3 
SCIT Student Handbook 
Contents 
Vision & Mission Statement 
Symbiosis  Infotech  Campus 
Health Care 
General Rules 
Student Identity Card & 
Dress Code 
Office Rules 
Eligibility for the Programme 
Payment of Fees 
Academic Rules and Guidelines 
Academic Standards & Guidelines 
Lab Rules 
Library Rules & Co-Curricular Activities 
Internship & Placement Eligibility Rules 
Discipline and Code of Conduct 
Committees 
Student Handbook 2025 
4 
Examination Rules 
SIU Grading Policy & Evaluation Policy 
ATKT Rules 
Rules of Revaluation 
Examination Fees


In [68]:
print(simple_rag("Where is Symbiosis Centre for Information Technology located?"))

Not available in handbook.


In [69]:
print(simple_rag("Provide the location details including Hinjawadi and Pune."))

The location details are as follows:

- Plot No. P-15, M.I.D.C, Symbiosis Infotech Campus, Rajiv Gandhi Infotech Park, Hinjawadi, Pune -411057.


In [70]:
print(simple_rag("What are the contact details mentioned in the handbook?"))

Not available in handbook.


In [71]:
for chunk in docs:
    if "vision" in chunk.page_content.lower():
        print(chunk.page_content)
        break

Student Handbook 2025 
Student Hand Book 2025 
MBA-ITBM 
Plot No. P-15, M.I.D.C, Symbiosis Infotech Campus, Rajiv Gandhi Infotech Park, 
Hinjawadi, Pune -411057.Tel No.: 020-22934308/09. 
Website: www.scit.edu
 2 
Student Handbook 2025 
3 
SCIT Student Handbook 
Contents 
Vision & Mission Statement 
Symbiosis  Infotech  Campus 
Health Care 
General Rules 
Student Identity Card & 
Dress Code 
Office Rules 
Eligibility for the Programme 
Payment of Fees 
Academic Rules and Guidelines 
Academic Standards & Guidelines 
Lab Rules 
Library Rules & Co-Curricular Activities 
Internship & Placement Eligibility Rules 
Discipline and Code of Conduct 
Committees 
Student Handbook 2025 
4 
Examination Rules 
SIU Grading Policy & Evaluation Policy 
ATKT Rules 
Rules of Revaluation 
Examination Fees


In [72]:
for chunk in docs:
    if "mission" in chunk.page_content.lower():
        print(chunk.page_content)
        break

Student Handbook 2025 
Student Hand Book 2025 
MBA-ITBM 
Plot No. P-15, M.I.D.C, Symbiosis Infotech Campus, Rajiv Gandhi Infotech Park, 
Hinjawadi, Pune -411057.Tel No.: 020-22934308/09. 
Website: www.scit.edu
 2 
Student Handbook 2025 
3 
SCIT Student Handbook 
Contents 
Vision & Mission Statement 
Symbiosis  Infotech  Campus 
Health Care 
General Rules 
Student Identity Card & 
Dress Code 
Office Rules 
Eligibility for the Programme 
Payment of Fees 
Academic Rules and Guidelines 
Academic Standards & Guidelines 
Lab Rules 
Library Rules & Co-Curricular Activities 
Internship & Placement Eligibility Rules 
Discipline and Code of Conduct 
Committees 
Student Handbook 2025 
4 
Examination Rules 
SIU Grading Policy & Evaluation Policy 
ATKT Rules 
Rules of Revaluation 
Examination Fees


In [73]:
for chunk in docs:
    if "vision" in chunk.page_content.lower() and len(chunk.page_content) > 200:
        print(chunk.page_content)
        print("\n------------------\n")

Student Handbook 2025 
Student Hand Book 2025 
MBA-ITBM 
Plot No. P-15, M.I.D.C, Symbiosis Infotech Campus, Rajiv Gandhi Infotech Park, 
Hinjawadi, Pune -411057.Tel No.: 020-22934308/09. 
Website: www.scit.edu
 2 
Student Handbook 2025 
3 
SCIT Student Handbook 
Contents 
Vision & Mission Statement 
Symbiosis  Infotech  Campus 
Health Care 
General Rules 
Student Identity Card & 
Dress Code 
Office Rules 
Eligibility for the Programme 
Payment of Fees 
Academic Rules and Guidelines 
Academic Standards & Guidelines 
Lab Rules 
Library Rules & Co-Curricular Activities 
Internship & Placement Eligibility Rules 
Discipline and Code of Conduct 
Committees 
Student Handbook 2025 
4 
Examination Rules 
SIU Grading Policy & Evaluation Policy 
ATKT Rules 
Rules of Revaluation 
Examination Fees

------------------

Committees 
Student Handbook 2025 
4 
Examination Rules 
SIU Grading Policy & Evaluation Policy 
ATKT Rules 
Rules of Revaluation 
Examination Fees 
Convocation Instructions 
for SCIT

In [74]:
print(simple_rag("Vision: Promoting international understanding through quality education"))

Promoting international understanding through quality education.


In [75]:
print(simple_rag("What is mentioned under Vision in the Student Handbook?"))

Promoting international understanding through quality education.


In [76]:
print(simple_rag("Provide the Vision and Mission statement as mentioned in the SCIT Student Handbook."))

Not available in handbook.


In [77]:
print(simple_rag("What is mentioned under Mission in the Student Handbook?"))

Students will communicate effectively that includes 
comprehension, presentation and reporting for business activities. 
Students will apply critical thinking to identify, analyze and 
formulate, business problems, evaluate problems and develop 
solutions. 
Students will develop an aptitude for research and lifelong learning. 
Students will recognize the need for innovation and entrepreneurial 
skills in the context of next generation technology leading towards 
national development.


In [78]:
print(simple_rag("What is the Symbiosis Centre of Health Care (SCHC)?"))

The Symbiosis Centre of Health Care (SCHC) was established on 14th June 1997 as an ‘In-house Health Care Unit’ of Symbiosis.


In [79]:
print(simple_rag("What is the vision of Symbiosis Centre of Health Care (SCHC)?"))

Envisioning a state of positive health in the community.


In [80]:
print(simple_rag("What healthcare services are provided by SCHC?"))

Preventive, promotive & curative healthcare services for the students and staff of the Symbiosis family.


In [81]:
print(simple_rag("What documents must students submit after reporting to campus?"))

A. Self-Attested Xerox Documents
1. Class 10 Marks List/Certificate
2. Class 12 Marks List/Certificate

B. Original Documents
1. SNAP Test Admit Card
2. SNAP Test Score Card
3. GEPIWAT Admit Card
4. TC/LC
5. Migration Certificate
6. Gap Certificate, if applicable
7. Anti-Ragging Affidavit (Student)
8. Anti-Ragging Affidavit (Parent)


In [82]:
print(simple_rag("Is part payment of fees allowed at SCIT?"))

No, part payment of fees is not allowed at SCIT.


In [83]:
print(simple_rag("What are the general behavior expectations under Campus Code?"))

General behavior must be polite and courteous to all at all time.
